# XGBoost 4-window vs 5-window tuned-model check

This notebook is a **stand-alone validation-stage check** for the two XGBoost
feature sets reported in the dissertation table:

- **4-window:** `{1, 7, 21, 30}`
- **5-window:** `{1, 7, 10, 21, 30}`

It reconstructs all prerequisites needed for this comparison from the two fixed
project data products:

1. `california_grid_centre_mask.csv`
2. `earthquake_california_grid_2010_2025_Mw25_centre_mask.csv`

It then:

1. reconstructs the daily $M_w^\ast\ge2.5$ count matrix;
2. constructs the $M_w^\ast\ge3.0$ future seven-day target;
3. reconstructs the chronological train/validation split;
4. reconstructs the smoothed long-term spatial forecast with
   $\varepsilon=0.10$;
5. constructs all seven raw XGBoost recent-seismicity windows;
6. refits the **already-selected tuned-best** 4-window and 5-window models;
7. checks whether the validation scores/iterations reproduce the original
   tuning results;
8. computes the paired forecast-origin score difference; and
9. runs moving-block bootstrap intervals with block lengths 7, 14 and 30 days.

## Important environment requirement

The original matched hyperparameter search was run with **XGBoost 3.4.1**.
The purpose of this notebook is to reproduce that comparison, so it deliberately
does **not** set `base_score` manually.

XGBoost 3.4.1 requires **Python 3.12 or later**.

The original rounded tuning results to recover are:

| Feature set | Validation log score | Best iteration |
|---|---:|---:|
| 1+7+21+30 | -0.036148 | 167 |
| 1+7+10+21+30 | -0.036150 | 190 |

This notebook does **not** repeat the 82-model hyperparameter search. It only
refits the two configurations already selected by that search.

In [ ]:
# ============================================================
# 0. Environment check
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import gammaln
import xgboost as xgb

pd.set_option("display.max_columns", 100)

EXPECTED_XGB_VERSION = "3.4.1"

print("Python:", sys.version)
print("XGBoost:", xgb.__version__)

if sys.version_info < (3, 12):
    raise RuntimeError(
        "This reproduction check requires Python >= 3.12 because "
        "XGBoost 3.4.1 requires Python >= 3.12."
    )

if xgb.__version__ != EXPECTED_XGB_VERSION:
    raise RuntimeError(
        f"Expected XGBoost {EXPECTED_XGB_VERSION}, found {xgb.__version__}. "
        "Use a Python 3.12+ environment and install xgboost==3.4.1, "
        "then restart the kernel."
    )

print("\nEnvironment matches the original XGBoost tuning run.")

## 1. Load the fixed project data products

This check starts from the **final harmonised modelling catalogue**, not from
the raw ComCat download. Magnitude harmonisation is therefore treated as an
upstream preprocessing step that has already been completed.

If the files are stored elsewhere, change `DATA_DIR`.

In [ ]:
# ============================================================
# 1. File paths and data loading
# ============================================================

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
AUDIT_DIR = ROOT / "outputs" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

GRID_MASK_FILE = DATA_DIR / "california_grid_centre_mask.csv"
FINAL_CATALOGUE_FILE = (
    DATA_DIR / "earthquake_california_grid_2010_2025_Mw25_centre_mask.csv"
)

for path in [GRID_MASK_FILE, FINAL_CATALOGUE_FILE]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path.resolve()}\n"
            "Put the file in the notebook folder or change DATA_DIR."
        )

grid_mask = pd.read_csv(GRID_MASK_FILE)
df = pd.read_csv(FINAL_CATALOGUE_FILE)

required_mask_cols = {"cell_id"}
required_catalogue_cols = {"time", "cell_id", "mag_Mw"}

missing_mask = required_mask_cols - set(grid_mask.columns)
missing_cat = required_catalogue_cols - set(df.columns)

if missing_mask:
    raise ValueError(f"Grid mask missing columns: {sorted(missing_mask)}")
if missing_cat:
    raise ValueError(f"Catalogue missing columns: {sorted(missing_cat)}")

grid_mask["cell_id"] = grid_mask["cell_id"].astype(int)
df["cell_id"] = df["cell_id"].astype(int)

df["time"] = pd.to_datetime(
    df["time"],
    format="mixed",
    utc=True,
    errors="raise"
)
df["date"] = df["time"].dt.floor("D").dt.tz_localize(None)

print("Final harmonised catalogue events:", len(df))
print("Retained grid cells:", len(grid_mask))
print("Occupied catalogue cells:", df["cell_id"].nunique())
print("Magnitude range:", df["mag_Mw"].min(), "to", df["mag_Mw"].max())

assert len(df) == 30347, f"Expected 30,347 events, found {len(df):,}"
assert len(grid_mask) == 1080, f"Expected 1,080 cells, found {len(grid_mask):,}"
assert grid_mask["cell_id"].is_unique

## 2. Reconstruct the daily $M_w^\ast\ge2.5$ grid counts

In [ ]:
# ============================================================
# 2. Daily grid-count matrix
# ============================================================

START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

dates = pd.date_range(START_DATE, END_DATE, freq="D")
cell_ids = np.sort(grid_mask["cell_id"].unique())

daily_counts_25 = (
    df.groupby(["date", "cell_id"])
      .size()
      .unstack(fill_value=0)
      .reindex(index=dates, columns=cell_ids, fill_value=0)
)

daily_counts_25.index.name = "date"

print("Number of days:", len(dates))
print("Number of retained cells:", len(cell_ids))
print("Daily count matrix shape:", daily_counts_25.shape)
print("Total Mw*>=2.5 events represented:", int(daily_counts_25.to_numpy().sum()))

assert daily_counts_25.shape == (5844, 1080)
assert int(daily_counts_25.to_numpy().sum()) == 30347
assert np.array_equal(
    np.asarray(daily_counts_25.columns, dtype=int),
    np.asarray(cell_ids, dtype=int)
)

## 3. Construct the $M_w^\ast\ge3.0$ future seven-day target

For each forecast origin $t$ and grid cell $g$,

\[
Y^{(7)}_{t,g}=\sum_{h=1}^{7}C^{(3.0)}_{t+h,g}.
\]

In [ ]:
# ============================================================
# 3. Target construction
# ============================================================

def build_daily_count_matrix(data, threshold, dates, cell_ids):
    """Daily cell counts for earthquakes satisfying mag_Mw >= threshold."""
    temp = data.loc[data["mag_Mw"] >= threshold].copy()

    return (
        temp.groupby(["date", "cell_id"])
            .size()
            .unstack(fill_value=0)
            .reindex(index=dates, columns=cell_ids, fill_value=0)
    )


def future_target_matrix(daily_counts, horizon):
    """
    Y_t = sum of counts from t+1 through t+horizon.
    Day t itself is excluded.
    """
    return (
        daily_counts.iloc[::-1]
        .rolling(window=horizon, min_periods=horizon)
        .sum()
        .shift(1)
        .iloc[::-1]
    )


TARGET_MAG = 3.0
FORECAST_HORIZON = 7

daily_counts_target = build_daily_count_matrix(
    df,
    threshold=TARGET_MAG,
    dates=dates,
    cell_ids=cell_ids
)

Y_7 = future_target_matrix(
    daily_counts_target,
    horizon=FORECAST_HORIZON
)

print("Target Mw*>=3.0 catalogue events:",
      int(daily_counts_target.to_numpy().sum()))
print("Target matrix shape:", Y_7.shape)
print("Complete target origins:", Y_7.dropna().shape[0])

assert int(daily_counts_target.to_numpy().sum()) == 6586
assert Y_7.shape == (5844, 1080)
assert Y_7.dropna().shape[0] == 5837

## 4. Construct all candidate recent-seismicity windows

XGBoost uses the **raw counts** for the recent-seismicity windows. The
long-term spatial forecast enters separately as `log_LT`.

In [ ]:
# ============================================================
# 4. Recent-seismicity windows
# ============================================================

candidate_windows = [1, 3, 7, 10, 14, 21, 30]

X_windows = {
    w: daily_counts_25.rolling(
        window=w,
        min_periods=w
    ).sum()
    for w in candidate_windows
}

for w in candidate_windows:
    print(f"X{w}:", X_windows[w].shape)

## 5. Reconstruct eligible forecast origins and temporal split

The 30-day history requirement determines the first eligible origin.
Seven-day gaps between subsets prevent target windows from crossing
train/validation/test boundaries.

In [ ]:
# ============================================================
# 5. Eligible origins and chronological split
# ============================================================

eligible_dates = (
    X_windows[30].notna().all(axis=1)
    &
    Y_7.notna().all(axis=1)
)

forecast_dates = dates[eligible_dates]

train_dates = forecast_dates[
    (forecast_dates >= "2010-01-30") &
    (forecast_dates <= "2019-12-24")
]

val_dates = forecast_dates[
    (forecast_dates >= "2020-01-01") &
    (forecast_dates <= "2022-12-24")
]

test_dates = forecast_dates[
    (forecast_dates >= "2023-01-01") &
    (forecast_dates <= "2025-12-24")
]

print("Eligible origins:",
      forecast_dates.min(), "to", forecast_dates.max(), "|", len(forecast_dates))
print("Train:", train_dates.min(), "to", train_dates.max(), "|", len(train_dates))
print("Validation:", val_dates.min(), "to", val_dates.max(), "|", len(val_dates))
print("Test:", test_dates.min(), "to", test_dates.max(), "|", len(test_dates))

assert len(forecast_dates) == 5808
assert len(train_dates) == 3616
assert len(val_dates) == 1089
assert len(test_dates) == 1089

## 6. Reconstruct the long-term spatial forecast

The long-term rate is estimated from the complete **2010-01-01 to
2019-12-31 training catalogue days**, exactly as in the original code.
The final spatial smoothing value is fixed at $\varepsilon=0.10$.

In [ ]:
# ============================================================
# 6. Long-term spatial-rate benchmark
# ============================================================

TRAIN_START = pd.Timestamp("2010-01-01")
TRAIN_END = pd.Timestamp("2019-12-31")

train_daily_target = daily_counts_target.loc[
    TRAIN_START:TRAIN_END
].copy()

n_train_days = len(train_daily_target)

N_g = train_daily_target.sum(axis=0).astype(float)
N_total = float(N_g.sum())
G = len(cell_ids)

p_empirical = N_g / N_total
weekly_total_rate = 7.0 * N_total / n_train_days

EPSILON_LT = 0.10

p_smoothed = (
    (1.0 - EPSILON_LT) * p_empirical
    + EPSILON_LT / G
)

lambda_LT_final = weekly_total_rate * p_smoothed
lambda_LT_reference = lambda_LT_final.copy()

print("Training calendar days:", n_train_days)
print("Training Mw*>=3.0 earthquakes:", N_total)
print("Weekly statewide LT rate:", weekly_total_rate)
print("LT min/max:", lambda_LT_reference.min(), lambda_LT_reference.max())
print("LT total:", lambda_LT_reference.sum())

assert n_train_days == 3652
assert N_total == 4565.0
assert np.isclose(weekly_total_rate, 8.75)
assert np.isclose(lambda_LT_reference.min(), 0.0008101851851851853)
assert np.isclose(lambda_LT_reference.max(), 1.1721409628412642)
assert np.isclose(lambda_LT_reference.sum(), 8.75)
assert (lambda_LT_reference > 0).all()

## 7. Flatten the targets and repeat the fixed LT feature

In [ ]:
# ============================================================
# 7. Flatten targets and LT feature
# ============================================================

def flatten_target(origin_dates):
    return (
        Y_7.loc[origin_dates]
        .to_numpy(dtype=float)
        .reshape(-1)
    )


def repeat_LT(origin_dates):
    return np.tile(
        lambda_LT_reference.to_numpy(dtype=float),
        len(origin_dates)
    )


y_train = flatten_target(train_dates)
y_val = flatten_target(val_dates)

LT_train = repeat_LT(train_dates)
LT_val = repeat_LT(val_dates)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("Training positive targets:", int(np.sum(y_train > 0)))
print("Training positive %:", 100 * np.mean(y_train > 0))
print("LT train min/max:", LT_train.min(), LT_train.max())

assert y_train.shape == (3905280,)
assert y_val.shape == (1176120,)
assert int(np.sum(y_train > 0)) == 14352
assert np.isclose(100 * np.mean(y_train > 0), 0.36750245821042277)

## 8. Build the XGBoost recent-feature matrices

This reproduces the exact row order and `float32` conversion used in the
original tuning notebook.

In [ ]:
# ============================================================
# 8. XGBoost feature preparation
# ============================================================

def build_xgb_recent_features(origin_dates):
    return np.column_stack([
        X_windows[w]
        .loc[origin_dates]
        .to_numpy(dtype=np.float32)
        .reshape(-1)
        for w in candidate_windows
    ]).astype(np.float32)


X_train_recent_xgb = build_xgb_recent_features(train_dates)
X_val_recent_xgb = build_xgb_recent_features(val_dates)

log_LT_train_xgb = np.log(
    LT_train.astype(np.float64)
).astype(np.float32)

log_LT_val_xgb = np.log(
    LT_val.astype(np.float64)
).astype(np.float32)

print("X_train_recent_xgb:", X_train_recent_xgb.shape)
print("X_val_recent_xgb:", X_val_recent_xgb.shape)
print("log_LT range:",
      log_LT_train_xgb.min(),
      log_LT_train_xgb.max())

assert X_train_recent_xgb.shape == (3905280, 7)
assert X_val_recent_xgb.shape == (1176120, 7)
assert np.isclose(log_LT_train_xgb.min(), -7.1182475, atol=1e-6)
assert np.isclose(log_LT_train_xgb.max(), 0.15883195, atol=1e-6)

## 9. Diagnostic checkpoint before fitting XGBoost

If this cell passes, the data state matches the original tuning notebook at
the point immediately before the matched feature-set comparison.

In [ ]:
# ============================================================
# 9. Data-state diagnostics
# ============================================================

checks = {
    "daily-count total = 30347":
        int(daily_counts_25.to_numpy().sum()) == 30347,
    "target event total = 6586":
        int(daily_counts_target.to_numpy().sum()) == 6586,
    "train origins = 3616":
        len(train_dates) == 3616,
    "validation origins = 1089":
        len(val_dates) == 1089,
    "cells = 1080":
        len(cell_ids) == 1080,
    "y_train observations = 3,905,280":
        len(y_train) == 3905280,
    "y_val observations = 1,176,120":
        len(y_val) == 1176120,
    "training positive targets = 14,352":
        int(np.sum(y_train > 0)) == 14352,
    "LT total = 8.75":
        np.isclose(lambda_LT_reference.sum(), 8.75),
    "LT min":
        np.isclose(
            lambda_LT_reference.min(),
            0.0008101851851851853
        ),
    "LT max":
        np.isclose(
            lambda_LT_reference.max(),
            1.1721409628412642
        )
}

diagnostic_table = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values())
})

display(diagnostic_table)

if not all(checks.values()):
    failed = [name for name, ok in checks.items() if not ok]
    raise RuntimeError(
        "Data-state diagnostic failed:\n- "
        + "\n- ".join(failed)
    )

print("All pre-XGBoost data-state checks passed.")

## 10. Build `DMatrix` objects for one window set

In [ ]:
# ============================================================
# 10. DMatrix construction
# ============================================================

def make_xgb_dmatrices(windows):

    cols = [
        candidate_windows.index(w)
        for w in windows
    ]

    Xtr = np.column_stack([
        log_LT_train_xgb,
        X_train_recent_xgb[:, cols]
    ]).astype(np.float32)

    Xva = np.column_stack([
        log_LT_val_xgb,
        X_val_recent_xgb[:, cols]
    ]).astype(np.float32)

    feature_names = (
        ["log_LT"]
        + [f"X{w}" for w in windows]
    )

    dtrain = xgb.DMatrix(
        Xtr,
        label=y_train,
        feature_names=feature_names
    )

    dval = xgb.DMatrix(
        Xva,
        label=y_val,
        feature_names=feature_names
    )

    return dtrain, dval

## 11. Tuned-best configurations from the original matched search

These values are **outputs of the already-completed 41-configuration matched
search**. We are not selecting them again here.

- 4-window: `log_LT + X1 + X7 + X21 + X30`
- 5-window: `log_LT + X1 + X7 + X10 + X21 + X30`

The learning rate remains fixed at `eta=0.05`.

In [ ]:
# ============================================================
# 11. Already-selected tuned-best configurations
# ============================================================

windows_4 = [1, 7, 21, 30]
windows_5 = [1, 7, 10, 21, 30]

best_4_params = {
    "max_depth": 4,
    "min_child_weight": 0.5,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 10.0,
    "reg_alpha": 1.0
}

best_5_params = {
    "max_depth": 5,
    "min_child_weight": 5.0,
    "subsample": 0.70,
    "colsample_bytree": 0.70,
    "reg_lambda": 0.1,
    "reg_alpha": 0.1
}

EXPECTED_4_SCORE_ROUNDED = -0.036148
EXPECTED_5_SCORE_ROUNDED = -0.036150
EXPECTED_4_ITERATION = 167
EXPECTED_5_ITERATION = 190

common_params = {
    "objective": "count:poisson",
    "eval_metric": "poisson-nloglik",
    "eta": 0.05,
    "tree_method": "hist",
    "seed": 2026,
    "nthread": -1
}

print("4-window:", windows_4, best_4_params)
print("5-window:", windows_5, best_5_params)

## 12. Refit only the two tuned-best models

Early stopping is retained so that we can check both the best validation
score and best iteration against the original tuning output.

**Do not add `base_score` manually in this reproduction run.**

In [ ]:
# ============================================================
# 12. Fit one tuned-best model and return validation predictions
# ============================================================

def fit_tuned_model(windows, tuned_params):

    dtrain, dval = make_xgb_dmatrices(windows)

    params = {
        **common_params,
        **tuned_params
    }

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    pred = model.predict(
        dval,
        iteration_range=(0, model.best_iteration + 1)
    )

    return {
        "model": model,
        "dval": dval,
        "pred": pred,
        "best_iteration": model.best_iteration + 1,
        "validation_nloglik": float(model.best_score),
        "validation_log_score_xgb": -float(model.best_score)
    }


result_4 = fit_tuned_model(
    windows_4,
    best_4_params
)

result_5 = fit_tuned_model(
    windows_5,
    best_5_params
)

print(
    "4-window | iteration:",
    result_4["best_iteration"],
    "| XGBoost validation log score:",
    result_4["validation_log_score_xgb"]
)

print(
    "5-window | iteration:",
    result_5["best_iteration"],
    "| XGBoost validation log score:",
    result_5["validation_log_score_xgb"]
)

## 13. Independent Poisson log-score calculation and reproduction check

The manual calculation uses

\[
s(y,\lambda)=y\log\lambda-\lambda-\log(y!).
\]

The rounded validation scores and best iterations should reproduce the
original tuning table before proceeding to the bootstrap.

In [ ]:
# ============================================================
# 13. Manual validation log scores
# ============================================================

def poisson_log_score_values(y, lam):
    y = np.asarray(y, dtype=np.float64)
    lam = np.clip(
        np.asarray(lam, dtype=np.float64),
        1e-15,
        None
    )

    return (
        y * np.log(lam)
        - lam
        - gammaln(y + 1.0)
    )


def mean_poisson_log_score(y, lam):
    return poisson_log_score_values(y, lam).mean()


pred_4 = result_4["pred"]
pred_5 = result_5["pred"]

score_4_manual = mean_poisson_log_score(y_val, pred_4)
score_5_manual = mean_poisson_log_score(y_val, pred_5)

comparison = pd.DataFrame([
    {
        "feature_set": "1+7+21+30",
        "best_iteration": result_4["best_iteration"],
        "xgb_validation_log_score":
            result_4["validation_log_score_xgb"],
        "manual_validation_log_score":
            score_4_manual,
        "expected_rounded_score":
            EXPECTED_4_SCORE_ROUNDED,
        "expected_iteration":
            EXPECTED_4_ITERATION
    },
    {
        "feature_set": "1+7+10+21+30",
        "best_iteration": result_5["best_iteration"],
        "xgb_validation_log_score":
            result_5["validation_log_score_xgb"],
        "manual_validation_log_score":
            score_5_manual,
        "expected_rounded_score":
            EXPECTED_5_SCORE_ROUNDED,
        "expected_iteration":
            EXPECTED_5_ITERATION
    }
])

display(comparison)

print(
    "\nObserved tuned-best difference (4-window - 5-window):",
    score_4_manual - score_5_manual
)

In [ ]:
# ============================================================
# 14. Stop if the original matched-search results are not recovered
# ============================================================

score4_ok = (
    round(result_4["validation_log_score_xgb"], 6)
    == EXPECTED_4_SCORE_ROUNDED
)

score5_ok = (
    round(result_5["validation_log_score_xgb"], 6)
    == EXPECTED_5_SCORE_ROUNDED
)

iter4_ok = (
    result_4["best_iteration"]
    == EXPECTED_4_ITERATION
)

iter5_ok = (
    result_5["best_iteration"]
    == EXPECTED_5_ITERATION
)

reproduction_checks = pd.DataFrame({
    "check": [
        "4-window rounded score",
        "5-window rounded score",
        "4-window best iteration",
        "5-window best iteration"
    ],
    "passed": [
        score4_ok,
        score5_ok,
        iter4_ok,
        iter5_ok
    ]
})

display(reproduction_checks)

if not reproduction_checks["passed"].all():
    raise RuntimeError(
        "The tuned-best models did not reproduce the original matched-search "
        "score/iteration outputs. Do NOT interpret the bootstrap below until "
        "this discrepancy is resolved."
    )

print(
    "Original matched-search outputs reproduced. "
    "Proceeding to paired bootstrap."
)

## 15. Forecast-origin paired score differences

Scores are averaged across the 1,080 grid cells at each validation forecast
origin. A **positive difference means the 4-window model is better**.

In [ ]:
# ============================================================
# 15. Forecast-origin score differences
# ============================================================

n_val_days = len(val_dates)
n_cells = len(cell_ids)

assert len(pred_4) == n_val_days * n_cells
assert len(pred_5) == n_val_days * n_cells
assert len(y_val) == n_val_days * n_cells

y_val_matrix = y_val.reshape(
    n_val_days,
    n_cells
)

pred_4_matrix = pred_4.reshape(
    n_val_days,
    n_cells
)

pred_5_matrix = pred_5.reshape(
    n_val_days,
    n_cells
)

score_4_by_day = (
    poisson_log_score_values(
        y_val_matrix,
        pred_4_matrix
    )
    .mean(axis=1)
)

score_5_by_day = (
    poisson_log_score_values(
        y_val_matrix,
        pred_5_matrix
    )
    .mean(axis=1)
)

d_4_minus_5 = (
    score_4_by_day
    - score_5_by_day
)

print("Mean difference (4 - 5):", d_4_minus_5.mean())
print("Median difference:", np.median(d_4_minus_5))
print(
    "4-window better on:",
    100 * np.mean(d_4_minus_5 > 0),
    "% of validation origins"
)
print(
    "Quantiles:",
    np.quantile(
        d_4_minus_5,
        [0, 0.05, 0.25, 0.5, 0.75, 0.95, 1]
    )
)

## 16. Moving-block bootstrap

The primary block length is 7 days because neighbouring seven-day targets
overlap. Block lengths 14 and 30 are sensitivity checks.

In [ ]:
# ============================================================
# 16. Moving-block bootstrap
# ============================================================

def moving_block_bootstrap_mean(
    x,
    block_length,
    n_boot=5000,
    seed=2026
):
    rng = np.random.default_rng(seed)

    x = np.asarray(x, dtype=np.float64)
    n = len(x)

    starts = np.arange(
        0,
        n - block_length + 1
    )

    n_blocks = int(
        np.ceil(n / block_length)
    )

    bootstrap_means = np.empty(
        n_boot,
        dtype=np.float64
    )

    for b in range(n_boot):

        sampled_starts = rng.choice(
            starts,
            size=n_blocks,
            replace=True
        )

        sample = np.concatenate([
            x[s:s + block_length]
            for s in sampled_starts
        ])[:n]

        bootstrap_means[b] = sample.mean()

    ci_lower, ci_upper = np.quantile(
        bootstrap_means,
        [0.025, 0.975]
    )

    return {
        "block_length": block_length,
        "mean_difference_4_minus_5":
            x.mean(),
        "ci_lower":
            ci_lower,
        "ci_upper":
            ci_upper,
        "p_difference_above_0":
            np.mean(bootstrap_means > 0),
        "ci_includes_zero":
            bool(ci_lower <= 0 <= ci_upper)
    }


bootstrap_results = pd.DataFrame([
    moving_block_bootstrap_mean(
        d_4_minus_5,
        block_length=b,
        n_boot=5000,
        seed=2026
    )
    for b in [7, 14, 30]
])

display(bootstrap_results)

## 17. Final decision check and saved outputs

If all three intervals include zero, the tuned-best comparison supports the
statement that there is **no clear evidence that adding the 10-day predictor
improves forecasting performance**.

If one or more intervals exclude zero, the dissertation wording should be
revisited rather than silently using the earlier screening-stage bootstrap.

In [ ]:
# ============================================================
# 17. Compact conclusion + save audit outputs
# ============================================================

all_include_zero = (
    bootstrap_results[
        "ci_includes_zero"
    ].all()
)

print("All 7/14/30-day CIs include zero:", all_include_zero)

if all_include_zero:
    print(
        "\nConclusion: the tuned-best paired bootstrap provides no clear "
        "evidence of a difference between the four-window and five-window "
        "models. Retaining the simpler {1,7,21,30} set is supported."
    )
else:
    print(
        "\nConclusion: at least one tuned-best bootstrap interval excludes "
        "zero. Revisit the dissertation wording before submission."
    )

comparison.to_csv(
    AUDIT_DIR / "xgb_4_vs_5_tuned_validation_scores.csv",
    index=False
)

bootstrap_results.to_csv(
    AUDIT_DIR / "xgb_4_vs_5_tuned_bootstrap.csv",
    index=False
)

np.savez_compressed(
    AUDIT_DIR / "xgb_4_vs_5_tuned_validation_predictions.npz",
    val_dates=val_dates.to_numpy(),
    cell_ids=np.asarray(cell_ids),
    y_val=y_val,
    pred_4=pred_4,
    pred_5=pred_5,
    score_4_by_day=score_4_by_day,
    score_5_by_day=score_5_by_day,
    difference_4_minus_5=d_4_minus_5
)

print("\nSaved:")
print(" - xgb_4_vs_5_tuned_validation_scores.csv")
print(" - xgb_4_vs_5_tuned_bootstrap.csv")
print(" - xgb_4_vs_5_tuned_validation_predictions.npz")

---

### Reference: earlier screening-stage comparison

The original notebook also performed a different 5-window vs 4-window
bootstrap using **provisional window-screening hyperparameters**. That
comparison gave approximately:

- 5-window score: `-0.036181861`
- 4-window score: `-0.036202397`
- mean difference (5 - 4): about `0.000021`

with 7-, 14- and 30-day bootstrap intervals all including zero.

That earlier result is **not** the target of this notebook. This notebook is
specifically designed to test the two **tuned-best models corresponding to
the dissertation table**.